# Predicting used-car asking prices in Sri Lanka

Which ensemble approach (bagging, boosting or stacking) best estimates the asking price of a newly listed car? Run the cells from top to bottom in Google Colab.

[Vehicle listings published by Prasad Nirmal](https://www.kaggle.com/datasets/prasadnirmal/srilankan-second-vehiclecar-price-dataset)

In [ ]:
%pip -q install kagglehub xgboost

In [ ]:
from pathlib import Path
import kagglehub
import pandas as pd

folder = Path(kagglehub.dataset_download('prasadnirmal/srilankan-second-vehiclecar-price-dataset'))
listings = pd.read_csv(folder / 'car_price_dataset.csv')

In [ ]:
print(f'{len(listings):,} listings, {listings.shape[1]} columns')
display(listings.head())
display(listings.isna().sum().rename('Missing values').to_frame())
listings = listings.drop(columns=['Unnamed: 0'], errors='ignore')
print('Exact repeats after removing the saved row number:', listings.duplicated().sum())

9,788 listings, 17 columns
   Unnamed: 0 Brand Model   YOM  ...  POWER STEERING POWER MIRROR POWER WINDOW  Price
0           0  AUDI    A1  2016  ...       Available    Available    Available  100.0
1           1  AUDI    A1  2017  ...       Available    Available    Available   97.0
2           2  AUDI    A1  2018  ...       Available    Available    Available   98.5
3           3  AUDI    A1  2017  ...       Available    Available    Available  107.0
4           4  AUDI    A1  2017  ...       Available    Available    Available   99.5

[5 rows x 17 columns]
                Missing values
Unnamed: 0                   0
Brand                        0
Model                        0
YOM                          0
Engine (cc)                  0
Gear                         0
Fuel Type                    0
Millage(KM)                  0
Town                         0
Date                         0
Leasing                      0
Condition                    0
AIR CONDITION                0


## First observations

- Each row is one advertisement. The first describes a 2016 Audi A1 in Gampaha with a 990 cc petrol engine and 99,000 km on the odometer.
- The target is the seller's asking price in `Price`. The Audi's recorded price is 100.0. The values look consistent with lakhs of Sri Lankan rupees, but the scale is unverified. All errors below use the original price units.
- A buyer could supply the brand, model, manufacture year, engine capacity, transmission, fuel type and mileage. These seven fields are the inputs for this first comparison.
- The inspection found zero missing values in all 17 original columns. Two Colombo listings in the preview share the same year, model and mileage but have different dates and prices. They may be reposts. We remove exact duplicate rows after dropping the saved row number. Without a vehicle identifier, we cannot reliably match every repost.

## Test on later listings

Keep the latest roughly 20% of dated listings for evaluation. All ads on the cutoff date go into the test set. This reflects the intended use of estimating asking prices for newly advertised cars.

In [ ]:
features = ['Brand', 'Model', 'YOM', 'Engine (cc)', 'Gear', 'Fuel Type', 'Millage(KM)']
cars = listings.drop_duplicates().copy()
cars['Date'] = pd.to_datetime(cars['Date'], errors='coerce')
cars['Price'] = pd.to_numeric(cars['Price'], errors='coerce')
cars = cars.loc[cars['Date'].notna() & cars['Price'].gt(0)].sort_values('Date')
cutoff = cars['Date'].iloc[int(0.80 * len(cars))]
train = cars.loc[cars['Date'] < cutoff]
test = cars.loc[cars['Date'] >= cutoff]
print(f'Training: {len(train):,} earlier listings')
print(f'Testing: {len(test):,} listings from {cutoff.date()} onward')

Training: 7,747 earlier listings
Testing: 2,023 listings from 2025-02-02 onward


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

categories = ['Brand', 'Model', 'Gear', 'Fuel Type']
models = {
    'Random Forest': RandomForestRegressor(n_estimators=150, min_samples_leaf=2, random_state=42, n_jobs=2),
    'XGBoost': XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                            objective='reg:squarederror', random_state=42, n_jobs=2),
}
predictions = pd.DataFrame(index=test.index)
fitted_models = {}
for name, model in models.items():
    encoder = ColumnTransformer([('categories', OneHotEncoder(handle_unknown='ignore'), categories)],
                                remainder='passthrough')
    pipeline = make_pipeline(encoder, model)
    pipeline.fit(train[features], train['Price'])
    fitted_models[name] = pipeline
    predictions[name] = pipeline.predict(test[features])

## Comparing three ensemble methods

Random Forest uses bagging: it averages trees trained on bootstrap samples. XGBoost uses boosting: each new tree adjusts the current predictions. For stacking, a ridge regression learns how to combine their estimates from out-of-fold predictions within the training period. The base pipelines are then fit on the whole training period for prediction. All three methods use identical inputs and test listings.

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold

paradigms = {'Random Forest': 'Bagging', 'XGBoost': 'Boosting', 'RF + XGBoost': 'Stacking'}
# Each date belongs to a single fold. The later test period stays outside all folds.
folds = list(GroupKFold(n_splits=4).split(train[features], groups=train['Date']))
stack = StackingRegressor(
    estimators=[('forest', fitted_models['Random Forest']), ('boost', fitted_models['XGBoost'])],
    final_estimator=Ridge(), cv=folds
)
stack.fit(train[features], train['Price'])
fitted_models['RF + XGBoost'] = stack
predictions['RF + XGBoost'] = stack.predict(test[features])

In [ ]:
scores = pd.Series({name: mean_absolute_error(test['Price'], predictions[name])
                    for name in predictions}, name='Mean absolute error').sort_values()
comparison = scores.round(2).to_frame()
comparison.insert(0, 'Paradigm', pd.Series(paradigms))
display(comparison)
best = scores.loc[list(fitted_models)].idxmin()
print(f'{best} has the lowest error on this split. '
      f'Its predictions differ from asking prices by {scores[best]:.2f} recorded units on average.')
examples = test[features + ['Price']].copy()
examples['Predicted'] = predictions[best]
examples['Absolute error'] = (examples['Price'] - examples['Predicted']).abs()
display(examples.nlargest(5, 'Absolute error').round(2))

               Paradigm  Mean absolute error
Random Forest   Bagging                 5.16
RF + XGBoost   Stacking                 5.46
XGBoost        Boosting                 6.02
Random Forest has the lowest error on this split. Its predictions differ from asking prices by 5.16 recorded units on average.
              Brand                 Model  ...  Predicted  Absolute error
1776  MERCEDES-BENZ  S400 LONG WHEEL BASE  ...     248.47          301.53
159             BYD                  SEAL  ...     193.97          200.53
1758  MERCEDES-BENZ          E300 BLUETEC  ...     211.14          151.39
1772  MERCEDES-BENZ                  S350  ...     152.72          107.28
130             BMW                  530E  ...     266.41           98.09

[5 rows x 10 columns]


Random Forest (bagging) had the smallest mean absolute error on the 2,023 later listings. The stack improved on XGBoost alone, but combining the two predictions did not improve on the forest in this test period. The ranking applies to these fitted models and this split.

## Dashboard models and feature importance

Shuffle one input at a time on the held-out listings. The increase in Random Forest error measures how much it depends on that input. Save all three trained models and the comparison for the dashboard.

In [ ]:
import joblib
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    fitted_models['Random Forest'], test[features], test['Price'],
    scoring='neg_mean_absolute_error', n_repeats=3, random_state=42
)
importance = pd.Series(importance.importances_mean, index=features).sort_values(ascending=False)
display(importance.round(2).rename('Increase in mean absolute error').to_frame())

# Start the dashboard controls with a real training listing for each brand and model.
catalog = {}
for (brand, model_name), group in train.groupby(['Brand', 'Model']):
    catalog.setdefault(brand, {})[model_name] = group[features].iloc[0].to_dict()

dashboard = {
    'models': fitted_models, 'paradigms': paradigms, 'catalog': catalog,
    'options': {field: sorted(train[field].unique()) for field in ['Gear', 'Fuel Type']},
    'years': (int(train['YOM'].min()), int(train['YOM'].max())),
    'scores': scores.to_dict(), 'importance': importance.to_dict(),
    'cutoff': str(cutoff.date()), 'train_count': len(train), 'test_count': len(test)
}
joblib.dump(dashboard, 'vehicle_models.joblib', compress=3)
print('Saved vehicle_models.joblib for the dashboard.')

             Increase in mean absolute error
YOM                                    13.01
Brand                                  11.06
Engine (cc)                             7.40
Millage(KM)                             6.80
Gear                                    5.72
Model                                   2.84
Fuel Type                               0.55
Saved vehicle_models.joblib for the dashboard.


## AI usage acknowledgment

ChatGPT helped find the dataset, draft the observations, and write the data inspection, preprocessing and model comparison code. ChatGPT also helped build the interactive dashboard and calculate permutation importance.